# GemmaCart – AI In-Store Shopping Assistant

> **Kaggle Gemma 4 Competition Submission**  
> Uses **Gemma 4** with function calling and vision to help Home Depot customers find products and navigate to their exact physical location — no GPS required.

---

## How It Works

1. **Auto check-in** — geofence detects the customer is at the store (GPS)
2. **Chat naturally** — *"Where can I find a cordless drill?"*
3. **Gemma 4 reasons** — calls `search_products` tool → gets real HD inventory
4. **Exact location** — *"Tools · Aisle 07, Bay 3 — walk right 7 aisles, enter aisle, walk halfway back"*
5. **Turn-by-turn nav** — calls `get_navigation_directions` → step-by-step walking instructions
6. **Multi-item route** — calls `plan_shopping_route` → optimised path through the store
7. **Photo search** — point camera at a product/label → Gemma vision identifies it
8. **Aisle self-update** — user reads orange overhead sign → app recalculates directions from current position

```
Mobile Browser (chat + nav UI)
    │
    ▼
FastAPI Backend
    │
    ▼
Gemma 4 (function calling + vision)
    │  tool calls
    ├─▶ search_products         → SQLite FTS5 (< 5 ms) or SerpAPI (live)
    ├─▶ get_navigation_directions → aisle-coordinate nav (no GPS needed)
    └─▶ plan_shopping_route     → nearest-neighbour TSP route
```

### Navigation Without GPS
Home Depot aisles are numbered on orange overhead signs — a physical coordinate system already built into every store.  
GemmaCart maps products to **Aisle N** (left-right) + **Bay N** (front-back depth), then generates  
human-readable turn-by-turn directions from wherever the customer currently is.


## 1. Setup

In [ ]:
!pip install -q fastapi uvicorn httpx python-multipart Pillow

import subprocess, sys, os, time

# Install Ollama if not present
result = subprocess.run(['which', 'ollama'], capture_output=True, text=True)
if not result.stdout.strip():
    print('Installing Ollama...')
    subprocess.run('curl -fsSL https://ollama.com/install.sh | sh', shell=True)

# Start Ollama server
subprocess.Popen(['ollama', 'serve'], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
time.sleep(3)
print('Ollama server started')

# Pull Gemma 4 (4b fits on Kaggle free tier; 12b for better accuracy if GPU available)
print('Pulling gemma4:4b...')
subprocess.run(['ollama', 'pull', 'gemma4:4b'])
print('Ready!')

## 2. Gemma 4 Function Calling — Tool Definitions

Three tools give Gemma 4 the ability to search inventory, generate turn-by-turn navigation, and plan multi-stop routes.  
Gemma decides autonomously which tools to call based on the customer's message.

In [ ]:
import httpx, json, asyncio, math

OLLAMA_URL = 'http://localhost:11434'
MODEL = 'gemma4:4b'

TOOLS = [
    {
        'type': 'function',
        'function': {
            'name': 'search_products',
            'description': (
                'Search the Home Depot store for products. '
                'Returns product names, prices, and exact aisle + bay location.'
            ),
            'parameters': {
                'type': 'object',
                'properties': {
                    'query': {'type': 'string', 'description': 'Product name or description'},
                    'max_results': {'type': 'integer', 'default': 6},
                },
                'required': ['query'],
            },
        },
    },
    {
        'type': 'function',
        'function': {
            'name': 'get_navigation_directions',
            'description': (
                'Generate step-by-step walking directions to a product using aisle numbers. '
                'No GPS needed — uses the orange overhead aisle signs inside the store. '
                'Call this after search_products to give the customer exact walking instructions.'
            ),
            'parameters': {
                'type': 'object',
                'properties': {
                    'product_id': {'type': 'string', 'description': 'ID of the product to navigate to'},
                    'from_aisle': {
                        'type': 'string',
                        'description': 'Current aisle number the customer is in (e.g. "05", "12", "G2"). Omit if at store entrance.',
                    },
                },
                'required': ['product_id'],
            },
        },
    },
    {
        'type': 'function',
        'function': {
            'name': 'plan_shopping_route',
            'description': 'Plan an optimised shopping route through the store for multiple products, minimising walking distance.',
            'parameters': {
                'type': 'object',
                'properties': {
                    'product_ids': {'type': 'array', 'items': {'type': 'string'}},
                },
                'required': ['product_ids'],
            },
        },
    },
]

SYSTEM = (
    'You are GemmaCart, an AI shopping assistant for The Home Depot in Dedham, MA. '
    'Help shoppers find products and walk to their exact location. '
    'Always call search_products before answering any product or location question. '
    'After finding a product, call get_navigation_directions to give precise walking instructions. '
    'For multiple items, call plan_shopping_route to optimise the path. '
    'Be brief and conversational — customers are on their phone mid-aisle.'
)

# ── Product catalogue (mirrors seeded SQLite DB) ─────────────────────────────
# Aisle format: "Department · Aisle XX, Bay N"
#   Aisle = left-right position (matches orange overhead signs, 1–35)
#   Bay   = front-back depth (Bay 1 = near checkout, Bay 6 = back wall)
MOCK_DB = {
    'drill': [
        {'id': 'p001', 'name': 'Milwaukee M18 FUEL 1/2" Drill/Driver Kit', 'brand': 'Milwaukee',
         'price': 379.00, 'aisle': 'Tools · Aisle 07, Bay 3', 'section': 'tools', 'gx': 3.2, 'gy': 5.5},
        {'id': 'p002', 'name': 'DEWALT 20V MAX Cordless Drill Kit', 'brand': 'DEWALT',
         'price': 199.00, 'aisle': 'Tools · Aisle 07, Bay 4', 'section': 'tools', 'gx': 3.2, 'gy': 4.5},
        {'id': 'p003', 'name': 'Ryobi ONE+ 18V Drill/Driver', 'brand': 'Ryobi',
         'price': 79.00, 'aisle': 'Tools · Aisle 08, Bay 1', 'section': 'tools', 'gx': 3.7, 'gy': 7.5},
    ],
    'paint': [
        {'id': 'p010', 'name': 'BEHR Premium Plus Interior Paint & Primer', 'brand': 'BEHR',
         'price': 39.98, 'aisle': 'Paint · Aisle 24, Bay 2', 'section': 'paint', 'gx': 11.0, 'gy': 6.5},
        {'id': 'p011', 'name': 'Wooster 9" Paint Roller Kit', 'brand': 'Wooster',
         'price': 12.98, 'aisle': 'Paint · Aisle 25, Bay 2', 'section': 'paint', 'gx': 11.4, 'gy': 6.5},
    ],
    'pvc': [
        {'id': 'p020', 'name': '1/2 in. x 10 ft. PVC Schedule 40 Pipe', 'brand': 'Charlotte Pipe',
         'price': 4.38, 'aisle': 'Plumbing · Aisle 14, Bay 2', 'section': 'plumbing', 'gx': 6.4, 'gy': 6.5},
        {'id': 'p021', 'name': '3/4 in. PVC Elbow Fitting (10-Pack)', 'brand': 'NIBCO',
         'price': 6.98, 'aisle': 'Plumbing · Aisle 14, Bay 5', 'section': 'plumbing', 'gx': 6.4, 'gy': 3.0},
    ],
    'led': [
        {'id': 'p030', 'name': 'Philips 60W Equiv LED Bulb (4-Pack)', 'brand': 'Philips',
         'price': 8.98, 'aisle': 'Lighting · Aisle 22, Bay 4', 'section': 'lighting', 'gx': 10.0, 'gy': 4.0},
    ],
    'lumber': [
        {'id': 'p040', 'name': '2 in. x 4 in. x 8 ft. Whitewood Stud', 'brand': 'Spruce',
         'price': 5.18, 'aisle': 'Lumber · Aisle 33, Bay 4', 'section': 'lumber', 'gx': 15.1, 'gy': 4.0},
        {'id': 'p041', 'name': '3/4 in. 4 ft. x 8 ft. Plywood', 'brand': 'Purebond',
         'price': 58.98, 'aisle': 'Lumber · Aisle 34, Bay 3', 'section': 'lumber', 'gx': 15.5, 'gy': 5.0},
    ],
}

# Flat lookup by product ID
_ALL_PRODUCTS = {p['id']: p for products in MOCK_DB.values() for p in products}

print('Tools and product catalogue ready.')
print(f'  {sum(len(v) for v in MOCK_DB.values())} products across {len(MOCK_DB)} categories')

## 3. Aisle-Coordinate Navigation Engine

Home Depot's physical orange overhead signs are a built-in coordinate system:  
- **Aisle number** (1–35) = left-right position across the store  
- **Bay number** (1–6) = front-to-back depth within an aisle  
  *(Bay 1 = near registers, Bay 6 = back wall)*

GemmaCart translates abstract grid positions `(gx, gy)` into these human-readable numbers,  
then generates step-by-step walking instructions anyone can follow — no GPS, no map needed.

In [ ]:
# ── Coordinate helpers ────────────────────────────────────────────────────────

GRID_COLS = 16   # store grid width
MAX_AISLE = 35   # HD stores have up to 35 numbered aisles
MAX_BAY   = 6    # bays per aisle (1 = front/registers, 6 = back wall)

def gx_to_aisle(gx: float) -> str:
    """Grid-x → HD aisle number string ('07', '14', 'G2').
    Garden section (gx < 2.0) uses 'G' prefix matching outdoor garden signs.
    """
    if gx < 2.0:
        g = max(1, min(4, int(gx * 2) + 1))
        return f'G{g}'
    n = max(1, min(MAX_AISLE, int((gx / GRID_COLS) * MAX_AISLE) + 1))
    return f'{n:02d}'

def gy_to_bay(gy: float) -> int:
    """Grid-y → bay number (Bay 1 = front/registers, Bay 6 = back wall).
    gy=8.5 is the entrance (front); gy=0 is the back wall.
    """
    normalized = 1.0 - min(1.0, max(0.0, gy / 8.5))
    return max(1, min(MAX_BAY, int(normalized * MAX_BAY) + 1))

def aisle_to_gx(aisle_str: str) -> float:
    """Reverse: HD aisle number string → grid-x coordinate."""
    s = str(aisle_str).strip().upper()
    if s.startswith('G'):
        n = int(s[1:]) if s[1:].isdigit() else 1
        return (max(1, min(4, n)) - 1) / 4 * 2.0
    n = int(s) if s.isdigit() else 0
    return (n - 1) * (GRID_COLS / MAX_AISLE) if n else 6.0


# ── Step-by-step direction builder ───────────────────────────────────────────

def build_nav_steps(from_gx: float, from_gy: float, product: dict) -> list[dict]:
    """Generate [{icon, text}] turn-by-turn instructions from any position to a product.
    
    Uses only aisle/bay numbers — directions a shopper can follow by reading overhead signs.
    """
    to_gx = product.get('gx', 8.0)
    to_gy = product.get('gy', 4.5)

    from_aisle = gx_to_aisle(from_gx)
    to_aisle   = gx_to_aisle(to_gx)
    to_bay     = gy_to_bay(to_gy)

    dx = to_gx - from_gx   # positive = right (higher aisle number)
    dy = to_gy - from_gy   # positive = toward front

    steps = []

    # Step 0 — enter store (only if starting from entrance)
    if from_gy >= 8.5:
        steps.append({'icon': '🚪', 'text': 'Enter through the main entrance'})

    # Step 1 — lateral: walk left or right to reach target aisle
    aisle_delta = abs(int(to_aisle.lstrip('G') or 0) - int(from_aisle.lstrip('G') or 0))
    if aisle_delta >= 1:
        direction = 'right' if dx > 0 else 'left'
        arrow     = '→' if dx > 0 else '←'
        steps.append({
            'icon': arrow,
            'text': f'Walk {direction} {aisle_delta} aisle{"s" if aisle_delta > 1 else ""} to reach Aisle {to_aisle}'
        })
    else:
        steps.append({'icon': '✓', 'text': f'You are in Aisle {to_aisle} — correct aisle!'})

    # Step 2 — depth: walk forward or backward to reach target bay
    from_bay = gy_to_bay(from_gy)
    bay_delta = abs(to_bay - from_bay)
    if bay_delta >= 1:
        toward    = 'back of the store' if to_bay > from_bay else 'front of the store'
        depth_dir = '↑' if to_bay > from_bay else '↓'
        steps.append({
            'icon': depth_dir,
            'text': f'Enter Aisle {to_aisle} and walk toward the {toward} to Bay {to_bay}'
        })
    else:
        steps.append({'icon': '✓', 'text': f'Bay {to_bay} — you are at the right depth'})

    # Step 3 — arrival
    dept = (product.get('aisle', '') or '').split('·')[0].strip()
    steps.append({
        'icon': '📦',
        'text': f'{product["name"].split(",")[0]} — {dept}, Aisle {to_aisle}, Bay {to_bay}'
    })

    return steps


# ── Tool executor (called by Gemma 4) ─────────────────────────────────────────

def execute_tool(name: str, args: dict) -> dict:
    if name == 'search_products':
        q = args.get('query', '').lower()
        limit = args.get('max_results', 6)
        # Simple keyword match against MOCK_DB keys
        results = next(
            (v for k, v in MOCK_DB.items() if k in q),
            MOCK_DB['drill']  # default demo fallback
        )
        return {'products': results[:limit], 'source': 'local_db', 'count': len(results[:limit])}

    if name == 'get_navigation_directions':
        pid = args.get('product_id', '')
        product = _ALL_PRODUCTS.get(pid)
        if not product:
            return {'error': f'Product {pid} not found'}
        # Default start: entrance at gx=6.0, gy=9.5
        from_aisle_str = args.get('from_aisle', '')
        from_gx = aisle_to_gx(from_aisle_str) if from_aisle_str else 6.0
        from_gy = 9.5 if not from_aisle_str else 4.5
        steps = build_nav_steps(from_gx, from_gy, product)
        to_aisle = gx_to_aisle(product['gx'])
        to_bay   = gy_to_bay(product['gy'])
        return {
            'product': product['name'],
            'destination': f'Aisle {to_aisle}, Bay {to_bay}',
            'steps': steps,
            'step_count': len(steps),
        }

    if name == 'plan_shopping_route':
        pids = args.get('product_ids', [])
        # Nearest-neighbour sort by aisle number
        items = [_ALL_PRODUCTS[p] for p in pids if p in _ALL_PRODUCTS]
        items.sort(key=lambda p: p.get('gx', 0))
        stops = [
            {'stop': i+1, 'product': p['name'], 'location': p['aisle'], 'product_id': p['id']}
            for i, p in enumerate(items)
        ]
        return {
            'stops': stops,
            'total_stops': len(stops),
            'estimated_minutes': max(5, len(stops) * 3),
            'strategy': 'left-to-right aisle order (minimises backtracking)',
        }

    return {'error': f'Unknown tool: {name}'}


print('Navigation engine ready.')

# Quick sanity check
p = _ALL_PRODUCTS['p001']
steps = build_nav_steps(6.0, 9.5, p)   # from entrance → Milwaukee drill
print(f'\nSample route: Entrance → {p["aisle"]}')
for s in steps:
    print(f'  {s["icon"]}  {s["text"]}')

## 4. Live Demo — Gemma 4 Reasoning Loop

Gemma 4 autonomously chains tool calls: search → navigate → route.  
Each turn shows which tool Gemma decided to call and the result it received.

In [ ]:
async def gemmacart_chat(user_message: str, from_aisle: str = '', verbose: bool = True) -> dict:
    """Send a message to GemmaCart and get product + navigation results.
    
    Args:
        user_message: Natural language query (e.g. 'Where can I find a cordless drill?')
        from_aisle:   Customer's current aisle (e.g. '05'). Empty = at store entrance.
        verbose:      Print reasoning trace.
    """
    # Inject current position into the initial message if provided
    ctx = f" (I'm currently in Aisle {from_aisle})" if from_aisle else ''
    messages = [{'role': 'user', 'content': user_message + ctx}]
    found_products, nav_result, final_reply = [], None, ''

    if verbose:
        print(f'Customer: {user_message}{ctx}')
        print('─' * 60)

    async with httpx.AsyncClient(timeout=120) as client:
        for turn in range(8):   # allow up to 8 tool-call turns
            resp = await client.post(f'{OLLAMA_URL}/api/chat', json={
                'model': MODEL,
                'messages': [{'role': 'system', 'content': SYSTEM}] + messages,
                'tools': TOOLS,
                'stream': False,
                'options': {'temperature': 0.15, 'num_predict': 512},
            })
            data = resp.json()
            msg = data.get('message', {})
            tool_calls = msg.get('tool_calls', [])

            if not tool_calls:
                final_reply = msg.get('content', '')
                break

            messages.append({'role': 'assistant', 'content': '', 'tool_calls': tool_calls})

            for tc in tool_calls:
                fn_name = tc['function']['name']
                fn_args = tc['function'].get('arguments', {})
                if isinstance(fn_args, str):
                    try:    fn_args = json.loads(fn_args)
                    except: fn_args = {}

                if verbose:
                    print(f'  [Turn {turn+1}] Gemma 4 → {fn_name}({json.dumps(fn_args)})')

                result = execute_tool(fn_name, fn_args)

                if fn_name == 'search_products':
                    found_products.extend(result.get('products', []))
                elif fn_name == 'get_navigation_directions':
                    nav_result = result

                messages.append({'role': 'tool', 'content': json.dumps(result)})

    if verbose:
        print(f'\nGemmaCart: {final_reply}')
        if found_products:
            print(f'\nProducts found ({len(found_products)}):')
            for p in found_products[:3]:
                print(f'  • {p["name"]:<50} ${p["price"]:>7.2f}  {p["aisle"]}')
        if nav_result and 'steps' in nav_result:
            print(f'\nNavigation to {nav_result["destination"]}:')
            for s in nav_result['steps']:
                print(f'  {s["icon"]}  {s["text"]}')

    return {'reply': final_reply, 'products': found_products, 'navigation': nav_result}

print('gemmacart_chat() ready.')

In [ ]:
# Demo 1: Simple location question from the store entrance
await gemmacart_chat('Where can I find a cordless drill?')

In [ ]:
# Demo 2: Budget constraint — Gemma filters and recommends
await gemmacart_chat('I need a drill but my budget is under $100. What do you recommend?')

In [ ]:
# Demo 3: Navigation from mid-store — customer is already in Aisle 12
# Gemma uses the customer's current position to give directions relative to where they are
await gemmacart_chat('I need LED bulbs', from_aisle='12')

In [ ]:
# Demo 4: Multi-item list — Gemma calls search + route tools
await gemmacart_chat('I need PVC pipe, paint, and a drill for a bathroom renovation project.')

## 5. Navigation Demo — Aisle Scan & Position Update

When a customer doesn't know their exact location, they look at the orange overhead sign and report their aisle.  
GemmaCart recalculates directions from their current position — no GPS, no map tap required.

In [ ]:
def demo_aisle_update(current_aisle: str, product_id: str):
    """Show how directions change when the customer reports their current aisle.
    
    This replicates the '📍 My Aisle' button in the mobile app:
    user reads the orange overhead sign → taps the number → directions recalculate.
    """
    product = _ALL_PRODUCTS.get(product_id)
    if not product:
        print(f'Product {product_id} not found')
        return

    from_gx = aisle_to_gx(current_aisle)
    from_gy = 4.5   # mid-aisle depth when user scans an aisle sign

    to_aisle = gx_to_aisle(product['gx'])
    to_bay   = gy_to_bay(product['gy'])

    steps = build_nav_steps(from_gx, from_gy, product)

    print(f'Customer location: Aisle {current_aisle}')
    print(f'Destination:       {product["aisle"]}')
    print(f'Product:           {product["name"]}')
    print('─' * 60)
    for i, s in enumerate(steps):
        marker = '▶' if i == 0 else ' '
        print(f'  {marker} {s["icon"]}  {s["text"]}')
    print()


# Scenario A: Customer just entered the store → drill
print('=== Scenario A: From store entrance ===')
demo_aisle_update('', 'p001')

# Scenario B: Customer is in Aisle 10 (lighting) and needs paint
print('=== Scenario B: Customer in Aisle 10 → Paint ===')
demo_aisle_update('10', 'p010')

# Scenario C: Customer is in Aisle 20 (back of store) → PVC pipe
print('=== Scenario C: Customer in Aisle 20 → PVC Pipe ===')
demo_aisle_update('20', 'p020')

# Scenario D: Customer is in garden section → indoor lumber
print('=== Scenario D: From Garden section (G2) → Lumber ===')
demo_aisle_update('G2', 'p040')

## 6. Product Database — SQLite FTS5 (1,378 Products)

The backend uses a SQLite FTS5 full-text search database seeded from SerpAPI.  
Search queries hit the DB first (< 5 ms), falling back to live SerpAPI only on cache miss.

In [ ]:
import sqlite3

DB_PATH = '../backend/products.db'
try:
    conn = sqlite3.connect(DB_PATH)
    conn.row_factory = sqlite3.Row

    total = conn.execute('SELECT COUNT(*) FROM products').fetchone()[0]
    print(f'Total products in DB: {total:,}')

    print('\nBy source:')
    for row in conn.execute('SELECT source, COUNT(*) n FROM products GROUP BY source ORDER BY n DESC'):
        print(f'  {row["source"]:20s}  {row["n"]:5d}')

    print('\nFTS5 search: "cordless drill" (store 2665)')
    rows = conn.execute("""
        SELECT p.name, p.brand, p.aisle, p.price
        FROM products p
        JOIN products_fts f ON f.id = p.id AND f.store_id = p.store_id
        WHERE products_fts MATCH '"cordless" OR "drill"' AND p.store_id = '2665'
        ORDER BY rank LIMIT 5
    """).fetchall()
    for r in rows:
        print(f'  {r["name"][:50]:<50}  ${r["price"] or 0:>7.2f}  {r["aisle"]}')

    conn.close()
except Exception as e:
    print(f'DB not seeded yet. Run backend/db_seeder.py to index products.')
    print(f'Error: {e}')
    print('Seeder indexes ~1,378 products from SerpAPI across 18 HD departments.')

## 7. Geofence — Store Auto Check-In

The mobile app uses the browser Geolocation API to detect when a customer enters a Home Depot.  
Within a 300 m radius, the app activates in-store mode — navigation and aisle tracking become available.

In [ ]:
import math

def haversine_km(lat1, lng1, lat2, lng2):
    R = 6371
    to_rad = math.radians
    a = (math.sin(to_rad(lat2 - lat1) / 2) ** 2
         + math.cos(to_rad(lat1)) * math.cos(to_rad(lat2))
         * math.sin(to_rad(lng2 - lng1) / 2) ** 2)
    return R * 2 * math.atan2(math.sqrt(a), math.sqrt(1 - a))

STORE = {'lat': 42.2478, 'lng': -71.1576, 'name': 'The Home Depot – Dedham, MA'}
GEOFENCE_KM = 0.3   # 300 m — roughly the footprint of a HD store + parking lot

customers = [
    (42.2480, -71.1574, 'Customer A — parking lot'),
    (42.2478, -71.1576, 'Customer B — inside store'),
    (42.2510, -71.1580, 'Customer C — across the street'),
    (42.3601, -71.0589, 'Customer D — downtown Boston'),
]

print(f'Store: {STORE["name"]}')
print(f'Geofence radius: {GEOFENCE_KM * 1000:.0f} m')
print('─' * 60)
for lat, lng, label in customers:
    dist = haversine_km(lat, lng, STORE['lat'], STORE['lng'])
    if dist <= GEOFENCE_KM:
        status = 'CHECKED IN ✓  → navigation activated'
    else:
        status = f'{dist * 0.621:.2f} mi away'
    print(f'  {label:<38}  {status}')

## 8. Summary

| Component | Technology | Role |
|-----------|-----------|------|
| **AI Model** | Gemma 4 (Ollama) | NL understanding, tool calling, multimodal vision |
| **Backend** | FastAPI (Python) | REST API, tool executor, route planner |
| **Database** | SQLite FTS5 | BM25 product search (< 5 ms), seeded from SerpAPI |
| **Live data** | SerpAPI | Real HD prices, stock, images on cache miss |
| **Navigation** | Aisle-coordinate engine | GPS-free turn-by-turn using overhead aisle signs |
| **Frontend** | Vanilla JS (mobile) | Chat UI, geofence, voice, camera, aisle scan |

### Why Gemma 4?
- **Function calling** — autonomously chains `search_products` → `get_navigation_directions` → `plan_shopping_route`
- **Vision** — processes product photos and identifies items by pointing the camera
- **Runs locally** — fully private, zero per-query cost after DB seeding
- **Conversational** — handles follow-ups, budget constraints, multi-item planning

### Navigation Without GPS
Every Home Depot aisle has an orange overhead sign with a number.  
GemmaCart turns those signs into a coordinate system:
- **Aisle 1–35** = left-right position across the store  
- **Bay 1–6** = front-to-back depth within an aisle  

The `get_navigation_directions` tool generates plain-English walking steps from any aisle to any product.  
Customers can tap "📍 My Aisle", enter the number on the sign above them, and get updated directions instantly.

### Performance
- **1,378 products** indexed across 18 HD departments
- **< 5 ms** search (SQLite FTS5) vs ~300 ms (live SerpAPI)
- **$0** per-query cost after one-time seeding
- **0 GPS calls** needed for in-store navigation

---
*GemmaCart — Kaggle Gemma 4 Competition Submission*